# Colab Part C: fine-tune Laya on the liquidation decisions (RQ5)

**J26-SE-325, Component 1.** Laya (`convaiinnovations/laya`) answers typed questions with
calibrated probabilities. This notebook fine-tunes it on the two-stage question set in
`decision/questions.py`, then analyses the result:

* **Stage 1 (analysis):** technical trend / momentum / support, fundamental health /
  valuation / earnings / distress, each from its own compact state.
* **Stage 2 (decision):** per holding sell now / spread / defer / exclude and downside risk;
  per portfolio the urgency and risk tolerance.

Labels come from **what actually happened next** (`decision/labels.py`), built only from
information available on each date, and split **by date**: the test set is the most recent
20% of dates. The encoder stays frozen, so the update that comes home is a few MB.

**RQ5:** *Does a fine-tuned decision model improve liquidation decisions over zero-shot and
rule-only?* Section 4 answers it against three references, not one: zero-shot Laya, the
majority class, and the base rate (climatology). Beating zero-shot alone is not enough; a
model must also beat simply predicting how often each outcome happened in training.

**Runtime:** T4 GPU. **Resumable:** every stage is saved on Drive and skipped when present.
**Prerequisites:** none. If Part A's `bars.parquet` is missing it is fetched here (Colab's
link makes that minutes). Part D's fundamentals backfill is optional; without it the
fundamental questions get no rows and that is reported, not hidden.


In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "laya==0.4.1 pypdf yfinance pandas-ta-openbb pyyaml requests pydantic pydantic-settings scikit-fuzzy"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


# In Colab always fetch: a re-run starts INSIDE the checkout, so find_root
# succeeded and pushed fixes never reached a live runtime.
root = None if IN_COLAB else find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
import laya

print("laya", laya.__version__)

## 1. Configuration

`RUN_SIZE` picks the preset. Each size writes to its own folder, so a smoke run never
overwrites a real one.

| Size | Scenarios | Stage 1 sampling | Epochs | CSE names fetched | T4 time (rough) |
|---|---|---|---|---|---|
| `smoke` | 200 | every 40th day | 1 | 20 | ~10-20 min |
| `pp1` | 3,000 | every 20th day | 2 | 60 | ~1-2 h |
| `full` | 20,000 | every 10th day | 3 | all | several hours |


In [ ]:
from pathlib import Path

RUN_SIZE = "pp1"  # "smoke" (10-20 min pipeline check) | "pp1" (1-2 h) | "full" (several hours)

PRESETS = {
    "smoke": dict(scenarios=200, sample_every=40, epochs=1, cse_limit=20),
    "pp1": dict(scenarios=3000, sample_every=20, epochs=2, cse_limit=60),
    "full": dict(scenarios=20000, sample_every=10, epochs=3, cse_limit=None),
}
CFG = PRESETS[RUN_SIZE]

DRIVE = Path("artifacts")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")

HYBRID_DIR = DRIVE / ("j26_hybrid" if IN_COLAB else "hybrid_training")
# CSE history beyond the endpoint's one-year window (built by `python -m data.cse_history`).
CSE_HISTORY = DRIVE / ("j26_cse_history" if IN_COLAB else "cse_history")
CSE_HISTORY = CSE_HISTORY if CSE_HISTORY.exists() else None
BARS = HYBRID_DIR / "bars.parquet"
BACKFILL = DRIVE / ("j26_se_325_backfill/fundamentals_backfill.parquet" if IN_COLAB
                    else "backfill/fundamentals_backfill.parquet")
OUT = DRIVE / ("j26_laya" if IN_COLAB else "laya_training") / RUN_SIZE
OUT.mkdir(parents=True, exist_ok=True)
HYBRID_DIR.mkdir(parents=True, exist_ok=True)
print("run size:", RUN_SIZE, CFG)
print("out:", OUT)
print("CSE history beyond one year:", CSE_HISTORY or "NOT FOUND (CSE uses the one-year endpoint window only)")
print("fundamentals backfill:", BACKFILL if BACKFILL.exists() else "NOT FOUND -- fundamental questions get no rows")

## 2. Market data

Reuses Part A's `bars.parquet` when present. Otherwise fetches CSE, the top Binance USDT
pairs and the US universe with the same code Part A uses, and saves it where Part A would,
so Part A later reuses it too.

In [ ]:
import pandas as pd

from experiments.train_hybrid_heads import collect_bars

bars = collect_bars(HYBRID_DIR, cse_limit=CFG["cse_limit"], cse_history=CSE_HISTORY)
dates = pd.to_datetime(bars["timestamp"])
print(f"{len(bars):,} rows | {bars['symbol'].nunique()} symbols | {dates.min().date()} to {dates.max().date()}")
bars.groupby("asset_class")["symbol"].nunique().rename("symbols").to_frame()

## 3. Train

Stage 1 rows, fine-tune #1 (encoder frozen), Stage 2 scenarios answered by the Stage 1 model
itself (no train/serve mismatch), fine-tune #2, the built-in evaluation and the head export.
Re-run this cell after a disconnect: finished stages are skipped.

In [ ]:
from experiments.train_laya import main as train_laya

args = ["--out", str(OUT), "--bars", str(BARS), "--scenarios", str(CFG["scenarios"]),
        "--sample-every", str(CFG["sample_every"]), "--epochs", str(CFG["epochs"])]
if BACKFILL.exists():
    args += ["--backfill", str(BACKFILL)]
train_laya(args)

## 4. RQ5 analysis

### 4.1 Data audit

Sizes, the date cutoff, and the label balance per question. Label balance matters: on a
question where 85% of outcomes are "false", 85% accuracy is what a constant guess gets.

In [ ]:
import json

import numpy as np

from decision import questions as Q


def read_jsonl(p):
    return [json.loads(line) for line in Path(p).read_text(encoding="utf-8").splitlines() if line.strip()]


split = json.loads((OUT / "split.json").read_text())
train_rows = read_jsonl(OUT / "decision_train.jsonl")  # includes the Stage 1 train rows (multi-task)
test_rows = read_jsonl(OUT / "stage1_test.jsonl") + read_jsonl(OUT / "decision_test.jsonl")


def stage_of(qid):
    for group, qs in Q.STAGE1.items():
        if qid in qs:
            return f"stage1_{group}"
    return "stage2_holding" if qid in Q.DECISION else "stage2_portfolio"


def gold_argmax(gold):
    return max(gold["probabilities"], key=gold["probabilities"].get)


train_dates = sorted(r["meta"]["date"] for r in train_rows)
test_dates = sorted(r["meta"]["date"] for r in test_rows)
print(f"train: {len(train_rows):,} rows, {train_dates[0]} to {train_dates[-1]}")
print(f"test:  {len(test_rows):,} rows, {test_dates[0]} to {test_dates[-1]}")
# train_laya.py splits Stage 1 and Stage 2 rows with SEPARATE date cutoffs, so the decision
# rows the final model trains on can run past the Stage 1 test cutoff. Any test row dated on
# or before the last training row shares market outcomes with training: it is excluded from
# every result below (it is still predicted, so the totals reconcile with evaluation.csv).
CLEAN_FROM = train_dates[-1]
overlap = sum(d <= CLEAN_FROM for d in test_dates)
if overlap:
    print(f"WARNING: {overlap:,} of {len(test_dates):,} test rows are dated on or before the last train row "
          f"({CLEAN_FROM}); they are excluded from the results. Clean test rows: {len(test_dates) - overlap:,}")
else:
    print("date split OK: every train row predates every test row")

audit = pd.DataFrame([{"split": name, "question": qid, "label": gold_argmax(g)}
                      for name, rows in (("train", train_rows), ("test", test_rows))
                      for r in rows for qid, g in r["gold"].items()])
counts = audit.groupby(["question", "split"]).size().unstack(fill_value=0)
counts["stage"] = [stage_of(q) for q in counts.index]
missing = sorted(set(Q.ALL_QUESTIONS) - set(counts.index))
print("questions with no rows:", missing or "none")
counts.sort_values("stage")

In [ ]:
# Majority share per question and split: the accuracy a constant guess gets.
balance = (audit.groupby(["question", "split"])["label"].value_counts(normalize=True)
           .rename("share").reset_index())
majority = (balance.sort_values("share", ascending=False).groupby(["question", "split"]).head(1)
            .pivot(index="question", columns="split", values=["label", "share"]))
majority.round(3)

### 4.2 Row-level predictions

The built-in `evaluation.csv` only keeps per-question means. Paired confidence intervals and
calibration need each row's prediction, so both models answer the test set once more,
batched by question set, and the predictions are cached on Drive.

In [ ]:
import gc


def predict_all(model_ref, rows, cache_path, batch_size=64):
    if cache_path.exists():
        return read_jsonl(cache_path)
    agent = laya.load(model_ref)
    groups = {}
    for i, r in enumerate(rows):
        groups.setdefault(json.dumps(r["questions"], sort_keys=True), []).append(i)
    answers = [None] * len(rows)
    for idx in groups.values():
        out = agent.predict_batch([rows[i]["state"] for i in idx], rows[idx[0]]["questions"], batch_size=batch_size)
        for i, o in zip(idx, out):
            answers[i] = o["answers"]
    del agent
    gc.collect()
    torch.cuda.empty_cache()
    with cache_path.open("w", encoding="utf-8") as fh:
        for a in answers:
            fh.write(json.dumps(a, default=float) + "\n")
    return answers


MODELS = {"zero_shot": "convaiinnovations/laya", "fine_tuned": str(OUT / "final")}
predictions = {name: predict_all(ref, test_rows, OUT / f"predictions_{name}.jsonl") for name, ref in MODELS.items()}
print({k: len(v) for k, v in predictions.items()})

In [ ]:
def predicted_probs(answer, qid):
    # Mirrors decision.labels.evaluate so the numbers reconcile with evaluation.csv.
    if "probabilities" in answer and Q.ALL_QUESTIONS[qid]["type"] == "choice":
        return {str(k): float(v) for k, v in answer["probabilities"].items()}
    if "noul" in answer:
        return {"true": float(answer["noul"]), "false": 1 - float(answer["noul"])}
    return {str(k): float(v) for k, v in (answer.get("probabilities") or {}).items()}


# Base-rate reference: the mean gold distribution per question over TRAIN rows only.
pooled = {}
for r in train_rows:
    for qid, g in r["gold"].items():
        for k, v in g["probabilities"].items():
            pooled.setdefault(qid, {}).setdefault(k, []).append(v)
climatology = {q: {k: float(np.mean(v)) for k, v in d.items()} for q, d in pooled.items()}
majority_label = {q: max(d, key=d.get) for q, d in climatology.items()}

records = []
for i, r in enumerate(test_rows):
    for qid, g in r["gold"].items():
        target = g["probabilities"]
        keys = list(target)
        truth = max(keys, key=target.get)
        rec = {"row": i, "question": qid, "stage": stage_of(qid), "date": r["meta"]["date"], "truth": truth,
               "majority_hit": float(majority_label.get(qid) == truth)}
        refs = {"climatology": climatology.get(qid, {})}
        for name, answers in predictions.items():
            a = (answers[i] or {}).get(qid)
            if a:
                refs[name] = predicted_probs(a, qid)
        for name, p in refs.items():
            if p:
                rec[f"{name}_hit"] = float(max(keys, key=lambda k: p.get(k, 0.0)) == truth)
                rec[f"{name}_brier"] = sum((p.get(k, 0.0) - target[k]) ** 2 for k in keys)
                rec[f"{name}_conf"] = max(p.get(k, 0.0) for k in keys)
        records.append(rec)
scored = pd.DataFrame(records).dropna(subset=["zero_shot_hit", "fine_tuned_hit", "climatology_brier"])
print(f"{len(scored):,} scored (row, question) pairs")

# Reconcile with the pipeline's own evaluation.csv: same rows, same scoring, so they must agree.
official = pd.read_csv(OUT / "evaluation.csv").pivot(index="question", columns="model", values="accuracy")
mine = scored.groupby("question")[["zero_shot_hit", "fine_tuned_hit"]].mean()
check = official.join(mine, how="inner")
gap = max((check["zero_shot"] - check["zero_shot_hit"]).abs().max(),
          (check["fine_tuned"] - check["fine_tuned_hit"]).abs().max())
print(f"max accuracy gap vs evaluation.csv: {gap:.4f}", "(OK)" if gap < 0.02 else "(INVESTIGATE before reporting)")

# From here on, only test rows strictly after the last training row (see 4.1).
scored = scored[scored["date"] > CLEAN_FROM].copy()
print(f"{len(scored):,} pairs on {scored['row'].nunique():,} clean test rows (after {CLEAN_FROM})")

### 4.3 Headline table

Per question: accuracy and Brier score (lower is better) for the majority class, the base
rate, zero-shot Laya and fine-tuned Laya. **Brier skill vs base rate** is
`1 - Brier(model) / Brier(base rate)`: above 0 means the model knows something the base rate
does not; at or below 0 means it does not, whatever its accuracy.

In [ ]:
def headline(frame):
    g = frame.groupby("question")
    t = pd.DataFrame({
        "stage": g["stage"].first(),
        "n": g.size(),
        "acc_majority": g["majority_hit"].mean(),
        "acc_zero_shot": g["zero_shot_hit"].mean(),
        "acc_fine_tuned": g["fine_tuned_hit"].mean(),
        "brier_base_rate": g["climatology_brier"].mean(),
        "brier_zero_shot": g["zero_shot_brier"].mean(),
        "brier_fine_tuned": g["fine_tuned_brier"].mean(),
    })
    t["skill_zero_shot"] = 1 - t["brier_zero_shot"] / t["brier_base_rate"]
    t["skill_fine_tuned"] = 1 - t["brier_fine_tuned"] / t["brier_base_rate"]
    return t.sort_values(["stage", "n"], ascending=[True, False])


table = headline(scored)
table.round(3)

### 4.4 Is the difference real? Paired bootstrap, by date

Fine-tuned minus zero-shot on the same test rows, resampled 2,000 times **by date, not by
row**. Every holding on a date shares that day's market move, and the labels look about a month
ahead, so rows are not independent draws; resampling them one at a time treats thousands of
correlated rows as thousands of observations and makes the intervals far too narrow. Resampling
whole dates makes the effective sample size the number of dates. A difference counts only when
its 95% interval excludes zero; a question with fewer than 10 test dates is reported as "too
few dates". With a short test period most intervals will be wide: that is the honest answer,
not a failure of the notebook.


In [ ]:
from evaluation.stats import paired_block_bootstrap


def paired_ci(frame, col_a, col_b):
    """(mean difference, lo, hi): rows on the same date are resampled together."""
    return paired_block_bootstrap(frame[col_a], frame[col_b], groups=frame["date"], n_boot=2000, seed=42)


rows = []
for qid, part in scored.groupby("question"):
    acc = paired_ci(part, "fine_tuned_hit", "zero_shot_hit")
    bri = paired_ci(part, "fine_tuned_brier", "zero_shot_brier")
    base = paired_ci(part, "fine_tuned_brier", "climatology_brier")
    if np.isnan(bri[0]):
        verdict = "too few dates"
    elif bri[2] < 0 and base[2] < 0:
        verdict = "better than zero-shot AND base rate"
    elif bri[2] < 0:
        verdict = "better than zero-shot, not base rate"
    elif bri[1] > 0:
        verdict = "WORSE than zero-shot"
    else:
        verdict = "no clear difference"
    rows.append({"question": qid, "stage": stage_of(qid), "n": len(part), "n_dates": part["date"].nunique(),
                 "acc_delta": acc[0], "acc_ci95": f"[{acc[1]:+.3f}, {acc[2]:+.3f}]",
                 "brier_delta": bri[0], "brier_ci95": f"[{bri[1]:+.3f}, {bri[2]:+.3f}]",
                 "brier_vs_base_rate": base[0], "verdict": verdict})
significance = pd.DataFrame(rows).sort_values(["stage", "question"])
significance.round(3)


### 4.5 Calibration

Laya's selling point is calibrated probabilities, so check it. Each prediction's confidence
(its top probability) is binned against how often it was right. Expected calibration error
(ECE) is the row-weighted gap between the two; lower is better.

In [ ]:
def reliability(frame, model, bins=10):
    conf, hit = frame[f"{model}_conf"].to_numpy(), frame[f"{model}_hit"].to_numpy()
    edges = np.linspace(0, 1, bins + 1)
    which = np.clip(np.digitize(conf, edges) - 1, 0, bins - 1)
    out, ece = [], 0.0
    for b in range(bins):
        m = which == b
        if m.any():
            out.append({"bin": f"{edges[b]:.1f}-{edges[b + 1]:.1f}", "n": int(m.sum()),
                        "mean_confidence": conf[m].mean(), "hit_rate": hit[m].mean()})
            ece += m.mean() * abs(conf[m].mean() - hit[m].mean())
    return pd.DataFrame(out), ece


calibration = {}
for model in ("zero_shot", "fine_tuned"):
    bins_table, calibration[model] = reliability(scored, model)
    print(f"{model}: ECE = {calibration[model]:.3f}")
    display(bins_table.round(3))

### 4.6 Robustness: embargo after the cutoff

Even after 4.1's exclusion, labels look up to about a month ahead, so test rows just after
the last training row share a price path with the last training labels. Re-scoring only test
rows dated at least 30 days after it removes that overlap. If the result holds here, it
is not an artefact of the boundary.

In [ ]:
EMBARGO_DAYS = 30
start = pd.Timestamp(CLEAN_FROM) + pd.Timedelta(days=EMBARGO_DAYS)
embargoed = scored[pd.to_datetime(scored["date"]) >= start]
print(f"embargoed test set: {len(embargoed):,} of {len(scored):,} pairs (from {start.date()})")
if len(embargoed) >= 50:
    cols = ["n", "acc_majority", "acc_zero_shot", "acc_fine_tuned", "skill_zero_shot", "skill_fine_tuned"]
    display(headline(embargoed)[cols].round(3))
else:
    print("too few rows after the embargo to say anything; report the main table with this caveat")

### 4.7 By stage

Stage 1 (analysis questions) and Stage 2 (decision questions) pooled. Stage 2 is what drives
the liquidation plan, so it is the part of RQ5 that matters most.

In [ ]:
by_stage = scored.groupby("stage").agg(
    n=("row", "size"),
    acc_majority=("majority_hit", "mean"),
    acc_zero_shot=("zero_shot_hit", "mean"),
    acc_fine_tuned=("fine_tuned_hit", "mean"),
    brier_base_rate=("climatology_brier", "mean"),
    brier_zero_shot=("zero_shot_brier", "mean"),
    brier_fine_tuned=("fine_tuned_brier", "mean"),
)
by_stage["skill_fine_tuned"] = 1 - by_stage["brier_fine_tuned"] / by_stage["brier_base_rate"]
by_stage.round(3)

### 4.8 Findings

Generated from the tables above, so it cannot say more than they do. Copy into the report
as is, or edit for tone; do not strengthen it.

In [ ]:
lines = [f"RQ5 ({RUN_SIZE} run, {scored['row'].nunique():,} clean test rows after {CLEAN_FROM}, "
         f"{scored['question'].nunique()} questions):"]
for verdict, n in significance["verdict"].value_counts().items():
    qs = ", ".join(significance.loc[significance["verdict"] == verdict, "question"])
    lines.append(f"  - {verdict}: {n} ({qs})")
skill = 1 - scored["fine_tuned_brier"].mean() / scored["climatology_brier"].mean()
lines.append(f"  - pooled accuracy: fine-tuned {scored['fine_tuned_hit'].mean():.3f}, "
             f"zero-shot {scored['zero_shot_hit'].mean():.3f}, majority class {scored['majority_hit'].mean():.3f}")
lines.append(f"  - pooled Brier skill vs base rate (fine-tuned): {skill:+.3f} -> "
             + ("adds information beyond the base rate" if skill > 0 else "does NOT beat the base rate"))
lines.append(f"  - calibration (ECE): zero-shot {calibration['zero_shot']:.3f}, fine-tuned {calibration['fine_tuned']:.3f}")
if missing:
    lines.append(f"  - not evaluated (no rows): {', '.join(missing)}")
if overlap:
    lines.append(f"  - {overlap:,} test rows overlapping the training period were excluded "
                 "(train_laya.py uses separate Stage 1 / Stage 2 cutoffs)")
if RUN_SIZE == "smoke":
    lines.append("  - SMOKE RUN: pipeline validation only; these numbers are not a result.")
findings = "\n".join(lines)
print(findings)
(OUT / "rq5_findings.txt").write_text(findings, encoding="utf-8")
table.round(4).to_csv(OUT / "rq5_table.csv")
significance.round(4).to_csv(OUT / "rq5_significance.csv", index=False)
by_stage.round(4).to_csv(OUT / "rq5_by_stage.csv")
print("saved to", OUT)

## 5. Bring the head home

On the laptop (the base checkpoint is already cached there):

```bash
python -m decision.checkpoint --install laya_head.zip
```

The decision layer applies an answer only at confidence >= 0.6, and the GA still computes
every amount. If 4.4 says the fine-tuned model is worse than zero-shot on the Stage 2
questions, do not install it; report that instead.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for name in ("laya_head.zip", "rq5_findings.txt", "rq5_table.csv", "rq5_significance.csv"):
        if (OUT / name).exists():
            files.download(str(OUT / name))